# Sentiment: LLM level (Qwen2.5-7B-Instruct, 4-bit, Kaggle GPU)
Prompting modes on the VnExpress dev comments: zero-shot, few-shot, few-shot with Chain-of-Thought, and
few-shot with the parent comment as context (ablation). Labels are the digits 0 (neg), 1 (neu), 2 (pos);
the scores are the softmax over those three tokens, so each item gets p_neg, p_neu, p_pos.

Few-shot without leakage (there is no public training set): items of fold k get 9 examples (3 per class)
drawn from the other four folds, same folds as `src/classifiers/sentiment.py`. The scored item is never
among its own examples.

Input: a private Kaggle dataset made from `data/tasks/sentiment/`. Settings: GPU T4 x2 or P100, Internet on.
To try prompts cheaply first, set `MODEL = "Qwen/Qwen2.5-1.5B-Instruct"` and `N_ITEMS = 100`.
Output (put in `results/sentiment/kaggle/`): `dev_scores_<tag>_{zero,few,cot,few_ctx}.csv` (columns id, pred,
p_neg, p_neu, p_pos, 4 decimals),
`<tag>_cot_outputs.jsonl` (reasoning text, for error analysis; `collect` counts outputs that switch to Chinese),
`<tag>_runs.json` (macro-F1, prediction shares and minutes per mode).

In [ ]:
!pip install -q -U bitsandbytes accelerate

In [ ]:
import functools, glob, json, os, re, time
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATA = os.path.dirname(glob.glob("/kaggle/input/**/dev.csv", recursive=True)[0])
OUT = "/kaggle/working"
MODEL = "Qwen/Qwen2.5-7B-Instruct"
TAG = "qwen" + MODEL.split("-")[1].lower()   # qwen7b, qwen1.5b
N_ITEMS = None        # None = all of dev; a small number for a quick prompt check
SHOTS_PER_CLASS = 3

CLASSES = ["neg", "neu", "pos"]
SEED = 42
read = lambda f: pd.read_csv(f"{DATA}/{f}", dtype=str, encoding="utf-8-sig", keep_default_na=False)
dev, corpus = read("dev.csv"), read("corpus.csv")
y = dev["label"].map(CLASSES.index).to_numpy()
mention = re.compile(r"@USER_[0-9a-f]+\s*:?", re.IGNORECASE)
clean = lambda t: mention.sub(" @user ", str(t)).strip()
# Same folds as src/classifiers/sentiment.py, so every item is scored by a model that never saw it
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(dev, y))
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(len(dev), dev.label.value_counts().to_dict(), len(corpus), gpu)

torch.manual_seed(SEED)
QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", quantization_config=QUANT).eval()
DIGITS = [tok.encode(d, add_special_tokens=False)[0] for d in "012"]   # 0 neg, 1 neu, 2 pos
fold_of = np.empty(len(dev), int)
for k, (_, te) in enumerate(folds):
    fold_of[te] = k

In [ ]:
def save_scores(ids, P, name):
    df = pd.DataFrame({"id": list(ids), "pred": [CLASSES[k] for k in P.argmax(1)]})
    for k, c in enumerate(CLASSES):
        df[f"p_{c}"] = P[:, k].round(4)
    df.to_csv(f"{OUT}/{name}.csv", index=False)

DEFINITION = """Bạn là người gán nhãn cảm xúc cho bình luận trên báo điện tử tiếng Việt. Gán theo thái độ của người viết bình luận, không theo việc tin được bàn là tin vui hay tin buồn.
0 = tiêu cực: chỉ trích, bức xúc, mỉa mai, lo lắng, buồn, chia buồn.
1 = trung tính: hỏi thông tin, nêu sự kiện, không rõ thái độ.
2 = tích cực: khen, ủng hộ, vui mừng, chúc mừng.
Trường hợp khó: mỉa mai bằng từ tích cực vẫn là 0. Đồng tình với một ý kiến thì xét nội dung được đồng tình. Có cả khen và chê thì chọn ý chiếm ưu thế.
"@user" là người được nhắc tên (đã ẩn danh)."""

def show(text, parent, ctx, limit=None):
    t = clean(text)[:limit] if limit else clean(text)
    if ctx and parent:
        p = clean(parent)[:limit] if limit else clean(parent)
        return f"Trả lời bình luận: {p}\nBình luận: {t}"
    return f"Bình luận: {t}"

@functools.lru_cache(maxsize=None)
def shots(k, ctx):
    """3 examples per class from the folds other than k (fixed per fold, so reproducible)."""
    pool = dev[fold_of != k]
    ex = pd.concat([pool[pool.label == c].sample(SHOTS_PER_CLASS, random_state=SEED + k) for c in CLASSES])
    ex = ex.sample(frac=1, random_state=SEED + k)
    return "\n\n".join(f"{show(r.text, r.parent_text, ctx, 300)}\nNhãn: {CLASSES.index(r.label)}" for r in ex.itertuples())

def messages(row, k, mode, ctx):
    k = int(k)
    system = DEFINITION
    if mode in ("few", "cot"):
        system += "\n\nVí dụ:\n\n" + shots(k, ctx)
    if mode == "cot":
        system += "\n\nHãy nêu ngắn gọn trong 2 đến 3 câu thái độ của người viết, rồi kết thúc bằng một dòng 'Nhãn: 0', 'Nhãn: 1' hoặc 'Nhãn: 2'."
        user = show(row.text, row.parent_text, ctx)
    else:
        system += "\n\nChỉ trả lời một chữ số 0, 1 hoặc 2."
        user = show(row.text, row.parent_text, ctx) + "\nNhãn:"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def prompt_ids(msgs):
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    return tok(text, add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)

@torch.no_grad()
def label_proba(ids):
    logits = model(ids).logits[0, -1]
    return torch.softmax(logits[DIGITS].float(), -1).cpu().numpy()

@torch.no_grad()
def score(row, k, mode, ctx):
    ids = prompt_ids(messages(row, k, mode, ctx))
    if mode != "cot":
        return label_proba(ids), None
    gen = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=256, do_sample=False)
    text = tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
    reasoning = text.split("Nhãn:")[0].rstrip()
    prefix = tok(reasoning + "\nNhãn: ", add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)
    return label_proba(torch.cat([ids, prefix], dim=1)), text

In [ ]:
items = dev if N_ITEMS is None else dev.groupby("label").sample(N_ITEMS // 3, random_state=SEED)
runs = []
for mode, ctx in (("zero", False), ("few", False), ("cot", False), ("few", True)):
    name = f"{TAG}_{mode}{'_ctx' if ctx else ''}"
    t0, P, texts = time.time(), [], []
    for i, row in zip(items.index, items.itertuples()):
        p, txt = score(row, fold_of[i], mode, ctx)
        P.append(p); texts.append(txt)
    P = np.array(P)
    if N_ITEMS is None:
        save_scores(items["id"], P, f"dev_scores_{name}")
    yt = items["label"].map(CLASSES.index).to_numpy()
    runs.append({"config": name, "n": len(items), "macro_f1": float(f1_score(yt, P.argmax(1), labels=[0, 1, 2], average="macro")),
                 "pred_share": (np.bincount(P.argmax(1), minlength=3) / len(P)).round(3).tolist(), "minutes": round((time.time() - t0) / 60, 1)})
    print(runs[-1])
    if mode == "cot":
        with open(f"{OUT}/{TAG}_cot_outputs.jsonl", "w") as f:
            for i, txt in zip(items["id"], texts):
                f.write(json.dumps({"id": i, "output": txt}, ensure_ascii=False) + "\n")
json.dump({"model": MODEL, "quant": "nf4 4-bit", "gpu": gpu, "shots_per_class": SHOTS_PER_CLASS, "seed": SEED,
           "runs": runs}, open(f"{OUT}/{TAG}_runs.json", "w"), indent=2)
pd.DataFrame(runs)